# Lesson 10: Final Project - Your Personal AI Tutor

## Putting It All Together!

Congratulations on making it to the final lesson! In this capstone project, you'll combine everything you've learned to build a complete **Personal AI Tutor** that can:

- Teach any topic you want to learn
- Adapt to your knowledge level
- Create lessons, quizzes, and exercises
- Track your progress

**This is your graduation project - let's build something amazing!**


---

## Skills You'll Use

This project combines EVERYTHING from the course:

| Lesson | Skill | How We'll Use It |
|--------|-------|-----------------|
| 2 | API Calls | Core communication with AI |
| 3 | Prompts | Crafting teaching prompts |
| 4 | Conversations | Multi-turn tutoring sessions |
| 5 | System Prompts | Creating the tutor personality |
| 6 | Temperature | Creative explanations vs factual answers |
| 7 | Structured Output | Quiz questions and assessments |
| 8 | Multi-Step | Creating lesson plans |
| 9 | Agents | Autonomous teaching decisions |


---

## Part 1: Setup


In [ ]:
# Setup - Run this first!

from dotenv import load_dotenv
from openai import OpenAI
import json
from IPython.display import Markdown, display

load_dotenv(override=True)
client = OpenAI()

def chat(prompt, temperature=0.7, system=None):
    """Chat with optional system prompt"""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content

def get_json(prompt, system=None):
    """Get JSON response"""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

print("✅ Ready to build the AI Tutor!")


---

## Part 2: The AI Tutor Class

Here's the complete AI Tutor that puts everything together!


In [ ]:
# THE COMPLETE AI TUTOR

class AITutor:
    """A personal AI tutor that teaches any topic"""
    
    def __init__(self, name="Professor AI"):
        self.name = name
        self.current_topic = None
        self.student_level = "beginner"
        self.lesson_history = []
        
        # The tutor's personality (System Prompt from Lesson 5!)
        self.personality = f"""
        You are {name}, an enthusiastic and patient teacher.
        
        Your teaching style:
        - Explain concepts simply before going into detail
        - Use analogies and real-world examples
        - Be encouraging and supportive
        - Adjust explanations based on student level
        - Ask questions to check understanding
        
        Current student level: {{level}}
        
        Always be helpful, never make the student feel bad for not knowing something.
        """
    
    def start_lesson(self, topic, level="beginner"):
        """Start a new lesson on a topic"""
        self.current_topic = topic
        self.student_level = level
        self.lesson_history = []
        
        print(f"📚 {self.name}: Starting Lesson on '{topic}'")
        print("=" * 60)
        
        # Step 1: Create a lesson plan (Multi-step from Lesson 8!)
        print("\n📝 Creating lesson plan...")
        lesson_plan = self._create_lesson_plan(topic, level)
        
        # Step 2: Deliver the introduction
        print("\n📖 Delivering introduction...")
        intro = self._deliver_introduction(topic, lesson_plan)
        
        self.lesson_history.append({
            "type": "lesson",
            "topic": topic,
            "plan": lesson_plan
        })
        
        return intro
    
    def _create_lesson_plan(self, topic, level):
        """Create a structured lesson plan (Structured Output from Lesson 7!)"""
        prompt = f"""
        Create a lesson plan for teaching "{topic}" to a {level} student.
        
        Return JSON with:
        {{
            "title": "lesson title",
            "learning_objectives": ["objective 1", "objective 2", "objective 3"],
            "key_concepts": ["concept 1", "concept 2", "concept 3"],
            "estimated_time": "X minutes",
            "sections": [
                {{"title": "section name", "points": ["point 1", "point 2"]}}
            ]
        }}
        
        Include 3 sections. Make it appropriate for the student level.
        """
        return get_json(prompt)
    
    def _deliver_introduction(self, topic, lesson_plan):
        """Deliver an engaging introduction"""
        prompt = f"""
        Deliver an engaging introduction for a lesson on: {topic}
        
        Lesson objectives: {lesson_plan['learning_objectives']}
        
        The introduction should:
        1. Hook the student's interest
        2. Explain why this topic matters
        3. Preview what they'll learn
        
        Keep it concise (2-3 paragraphs). Make it exciting!
        """
        return chat(prompt, temperature=0.7, 
                   system=self.personality.format(level=self.student_level))
    
    def teach_concept(self, concept):
        """Teach a specific concept"""
        print(f"\n📚 Teaching: {concept}")
        print("-" * 40)
        
        prompt = f"""
        Teach this concept to a {self.student_level} student: {concept}
        
        Include:
        1. Simple explanation
        2. A helpful analogy
        3. A practical example
        4. A quick check-in question
        
        Make it engaging and easy to understand.
        """
        explanation = chat(prompt, temperature=0.6,
                          system=self.personality.format(level=self.student_level))
        
        self.lesson_history.append({
            "type": "concept",
            "concept": concept
        })
        
        return explanation
    
    def quiz(self, num_questions=3):
        """Generate a quiz on the current topic (Structured Output!)"""
        if not self.current_topic:
            return "Please start a lesson first!"
        
        print(f"\n📝 Quiz Time: {self.current_topic}")
        print("=" * 40)
        
        prompt = f"""
        Create a {num_questions}-question quiz about: {self.current_topic}
        Difficulty: {self.student_level}
        
        Return JSON with:
        {{
            "questions": [
                {{
                    "question": "the question",
                    "options": ["A) option1", "B) option2", "C) option3", "D) option4"],
                    "correct": "A",
                    "explanation": "why this is correct"
                }}
            ]
        }}
        """
        quiz_data = get_json(prompt)
        
        # Run the interactive quiz
        score = 0
        for i, q in enumerate(quiz_data['questions'], 1):
            print(f"\nQuestion {i}: {q['question']}")
            for opt in q['options']:
                print(f"   {opt}")
            
            answer = input("Your answer (A/B/C/D): ").strip().upper()
            
            if answer == q['correct']:
                print("✅ Correct!")
                score += 1
            else:
                print(f"❌ Not quite. The answer is {q['correct']}")
            print(f"   Explanation: {q['explanation']}")
        
        print(f"\n📊 Score: {score}/{num_questions}")
        
        if score == num_questions:
            print("🎉 Perfect! You've mastered this topic!")
        elif score >= num_questions / 2:
            print("👍 Good job! Keep practicing!")
        else:
            print("📚 Let's review this topic together!")
        
        return score
    
    def ask(self, question):
        """Ask the tutor a question (Conversation from Lesson 4!)"""
        print(f"\n❓ Your question: {question}")
        print("-" * 40)
        
        context = f"Current topic: {self.current_topic}" if self.current_topic else ""
        
        prompt = f"""
        {context}
        Student question: {question}
        
        Answer helpfully and check if they understood.
        """
        
        answer = chat(prompt, temperature=0.5,
                     system=self.personality.format(level=self.student_level))
        
        return answer
    
    def summarize(self):
        """Summarize what was learned"""
        if not self.current_topic:
            return "No lesson to summarize!"
        
        print(f"\n📋 Lesson Summary: {self.current_topic}")
        print("=" * 40)
        
        prompt = f"""
        Create a brief summary of the lesson on: {self.current_topic}
        
        Include:
        1. Key points learned (bullet points)
        2. One thing to remember
        3. Suggested next topic to learn
        
        Keep it concise and actionable.
        """
        
        return chat(prompt, temperature=0.5,
                   system=self.personality.format(level=self.student_level))

print("✅ AITutor class created!")


---

## Part 3: Using Your AI Tutor

Let's put the tutor to work!


In [ ]:
# Create your tutor
tutor = AITutor(name="Professor Ada")

# Start a lesson
intro = tutor.start_lesson("Machine Learning Basics", level="beginner")
print("\n")
display(Markdown(intro))


In [ ]:
# Learn a specific concept
explanation = tutor.teach_concept("What is a neural network?")
display(Markdown(explanation))


In [ ]:
# Ask a question
answer = tutor.ask("How is machine learning different from regular programming?")
display(Markdown(answer))


In [ ]:
# Take a quiz!
# Note: This is interactive - you'll need to type your answers
score = tutor.quiz(num_questions=2)


In [ ]:
# Get a summary
summary = tutor.summarize()
display(Markdown(summary))


---

## Part 4: Try Your Own Topic!

Now try using the tutor for any topic YOU want to learn!


In [ ]:
# YOUR TURN: Learn any topic you want!

# Create a new tutor with a custom name
my_tutor = AITutor(name="Your Teacher Name")

# Choose your topic and level
my_topic = "Your Topic Here"  # Change this!
my_level = "beginner"  # Options: beginner, intermediate, advanced

# Start learning!
# intro = my_tutor.start_lesson(my_topic, level=my_level)
# display(Markdown(intro))


---

## Congratulations! You've Completed the Course!

### What You've Learned

1. **AI Fundamentals** - What AI and LLMs are
2. **API Basics** - Making AI calls programmatically
3. **Prompt Engineering** - Writing effective prompts
4. **Conversations** - Building memory and context
5. **System Prompts** - Creating AI personalities
6. **Temperature** - Controlling creativity
7. **Structured Output** - Getting JSON and formatted data
8. **Multi-Step Workflows** - Chaining AI calls
9. **AI Agents** - Building autonomous AI systems
10. **Complete Applications** - Putting it all together!

### What You've Built

- Joke Generator
- Story Generator
- Memory Bot
- AI Personalities (Pirate, Teacher, Chef)
- Temperature Comparison Tool
- Recipe Extractor
- Blog Post Generator
- Research Assistant Agent
- **Personal AI Tutor!**

### What's Next?

Your AI journey is just beginning! Here are some ideas:

1. **Enhance the Tutor** - Add more features, track progress over time
2. **Build Your Own Projects** - Customer service bots, content generators
3. **Learn Advanced Topics** - Function calling, RAG, fine-tuning
4. **Explore Frameworks** - LangChain, CrewAI, AutoGen

### Final Thoughts

Remember:
- AI is a tool - learn to use it effectively
- Start simple, iterate and improve
- The best way to learn is by building
- Stay curious and keep experimenting!

**You did it! Welcome to the world of AI development!** 🎓🚀
